[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/12_montecarlo/notebooks/aplicaciones/04_inferencia_bayesiana.ipynb)

# Aplicación 4: Inferencia Bayesiana Aproximada

## Conexión con el curso

En el **módulo 05** (Probabilidad) aprendimos el Teorema de Bayes:

$$p(\theta \mid \text{datos}) = \frac{p(\text{datos} \mid \theta) \cdot p(\theta)}{p(\text{datos})}$$

En el **módulo 10** (Redes Bayesianas) vimos cómo hacer inferencia **exacta** en grafos con tablas de probabilidad finitas.

Pero en muchos modelos reales, el posterior $p(\theta \mid \text{datos})$ **no tiene forma cerrada**. El denominador $p(\text{datos}) = \int p(\text{datos} \mid \theta) p(\theta)\, d\theta$ es intratable.

**Monte Carlo** es la respuesta: en lugar de calcular el posterior exacto, lo **aproximamos con muestras**.

Este notebook explora dos enfoques: **rejection sampling** (el más simple, y el que muestra los límites) e **importance sampling** para posteriors (el más eficiente de los dos).

In [ ]:
# Descomenta si estás en Colab
# !pip install numpy matplotlib scipy --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm, beta as beta_dist
from scipy.special import expit  # sigmoid

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

rng = np.random.default_rng(42)
print("Listo ✓")

---
## Parte 1: El Modelo — Posterior No Conjugado

### Regresión Logística Bayesiana

Tenemos datos $(x_i, y_i)$ donde $y_i \in \{0, 1\}$ (binario) y queremos modelar:

$$P(y_i = 1 \mid x_i, \theta) = \sigma(\theta_0 + \theta_1 x_i), \qquad \sigma(z) = \frac{1}{1+e^{-z}}$$

Con prior $\theta = (\theta_0, \theta_1) \sim \mathcal{N}(0, \tau^2 I)$.

**El posterior:**

$$p(\theta \mid \mathbf{y}, \mathbf{x}) \propto p(\mathbf{y} \mid \mathbf{x}, \theta) \cdot p(\theta) = \prod_i \sigma(\theta^T x_i)^{y_i}(1-\sigma(\theta^T x_i))^{1-y_i} \cdot \mathcal{N}(\theta; 0, \tau^2 I)$$

La likelihood es logística → **no existe forma cerrada para el posterior**. No hay conjugación. Necesitamos aproximación numérica.

In [ ]:
# Generate synthetic data from a known theta_true
np.random.seed(42)
n_data = 30  # <-- CHANGE THIS to see how more data sharpens the posterior
theta_true = np.array([0.5, 2.0])  # true [intercept, slope]

x_data = np.random.uniform(-2, 2, n_data)
logit_p = theta_true[0] + theta_true[1] * x_data
y_data = np.random.binomial(1, expit(logit_p))

print(f"Generados {n_data} observaciones con θ_true = {theta_true}")
print(f"Proporción de y=1: {y_data.mean():.2f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(x_data[y_data==0], y_data[y_data==0], s=60, color="#2E86AB", alpha=0.7, label="$y=0$")
ax.scatter(x_data[y_data==1], y_data[y_data==1], s=60, color="#E94F37", alpha=0.7, label="$y=1$")
x_range = np.linspace(-2, 2, 200)
ax.plot(x_range, expit(theta_true[0] + theta_true[1]*x_range),
        "k--", lw=1.5, label="Probabilidad verdadera")
ax.set_xlabel("$x$")
ax.set_ylabel("$y$")
ax.set_title("Datos de regresión logística")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Define log posterior (unnormalized)
tau = 3.0  # prior std  # <-- CHANGE THIS

def log_likelihood(theta, x, y):
    """Log-likelihood of logistic regression."""
    logit_p = theta[0] + theta[1] * x
    log_p = np.log(expit(logit_p) + 1e-15)
    log_1mp = np.log(1 - expit(logit_p) + 1e-15)
    return np.sum(y * log_p + (1 - y) * log_1mp)

def log_prior(theta, tau=3.0):
    """Log prior: N(0, tau^2 I)."""
    return -0.5 * np.sum(theta**2) / tau**2

def log_posterior(theta, x, y, tau=3.0):
    return log_likelihood(theta, x, y) + log_prior(theta, tau)

# Visualize the posterior over a 2D grid (ground truth)
theta0_range = np.linspace(-3, 3, 100)
theta1_range = np.linspace(-1, 5, 100)
T0, T1 = np.meshgrid(theta0_range, theta1_range)

log_post_grid = np.array([
    log_posterior(np.array([t0, t1]), x_data, y_data, tau)
    for t0, t1 in zip(T0.ravel(), T1.ravel())
]).reshape(T0.shape)

# Normalize for visualization
post_grid = np.exp(log_post_grid - log_post_grid.max())

fig, ax = plt.subplots(figsize=(7, 6))
cf = ax.contourf(T0, T1, post_grid, levels=20, cmap="Blues")
ax.plot(*theta_true, "r*", ms=15, label=f"$\\theta_{{\\text{{true}}}} = {theta_true}$")
plt.colorbar(cf, ax=ax, label="Posterior (renormalizado)")
ax.set_xlabel("$\\theta_0$ (intercepto)")
ax.set_ylabel("$\\theta_1$ (pendiente)")
ax.set_title("Posterior $p(\\theta \\mid \\mathbf{y})$ (no tiene forma cerrada)")
ax.legend()
plt.tight_layout()
plt.show()

---
## Parte 2: Rejection Sampling

### La idea

Queremos muestras de $p(\theta \mid \mathbf{y})$ pero no podemos muestrear directamente.

**Rejection sampling** usa una distribución **envolvente** $q(\theta)$ tal que:

$$M \cdot q(\theta) \geq p^{∗}(\theta) \quad \forall \theta$$

donde $p^{∗}$ es el posterior sin normalizar.

**Algoritmo:**
1. Muestrea $\theta \sim q(\theta)$
2. Muestrea $u \sim \text{Uniform}(0, 1)$
3. Si $u \leq p^{∗}(\theta) / (M \cdot q(\theta))$: **acepta** $\theta$
4. Si no: **rechaza** y vuelve al paso 1

Las muestras aceptadas son exactamente de $p(\theta \mid \mathbf{y})$.

In [ ]:
def rejection_sampling_2d(log_post_fn, proposal_mean, proposal_std, M_scale, n_accept, seed=None):
    """Rejection sampling from a 2D distribution."""
    rng_local = np.random.default_rng(seed)
    accepted = []
    n_total = 0

    # log M: chosen so M*q >= p* (use max of log_posterior from grid + small buffer)
    log_M = log_post_grid.max() + np.log(M_scale)

    while len(accepted) < n_accept:
        # Sample from proposal: N(proposal_mean, proposal_std^2 * I)
        theta = rng_local.normal(proposal_mean, proposal_std, size=2)

        # Log acceptance probability
        log_post = log_post_fn(theta, x_data, y_data, tau)
        log_q = norm.logpdf(theta[0], proposal_mean[0], proposal_std) + \
                norm.logpdf(theta[1], proposal_mean[1], proposal_std)
        log_alpha = log_post - (log_M + log_q)

        # Accept/reject
        if np.log(rng_local.uniform()) < log_alpha:
            accepted.append(theta)
        n_total += 1
        if n_total > 2_000_000:
            print("Warning: too many rejections, stopping early")
            break

    acceptance_rate = len(accepted) / n_total
    return np.array(accepted), acceptance_rate

# Proposal: N(prior_mean, prior_std) — wide envelope
proposal_mean = np.array([0.0, 1.0])  # rough guess of posterior mode
proposal_std = 2.0

samples, acc_rate = rejection_sampling_2d(
    log_posterior, proposal_mean, proposal_std,
    M_scale=1.5, n_accept=2000, seed=42
)

print(f"Tasa de aceptación: {acc_rate:.4f} ({acc_rate*100:.2f}%)")
print(f"Muestras obtenidas: {len(samples)}")
print(f"Media posterior θ₀: {samples[:,0].mean():.3f} (verdadero: {theta_true[0]})")
print(f"Media posterior θ₁: {samples[:,1].mean():.3f} (verdadero: {theta_true[1]})")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].contourf(T0, T1, post_grid, levels=15, cmap="Blues", alpha=0.5)
axes[0].scatter(samples[:,0], samples[:,1], s=3, alpha=0.3, color="#E94F37", label="Muestras RS")
axes[0].plot(*theta_true, "k*", ms=15, label="θ_true")
axes[0].set_xlabel("$\\theta_0$")
axes[0].set_ylabel("$\\theta_1$")
axes[0].set_title(f"Rejection Sampling — tasa: {acc_rate:.2%}")
axes[0].legend()

for j, (samples_j, name) in enumerate(zip(samples.T, ["$\\theta_0$", "$\\theta_1$"])):
    axes[1].hist(samples_j, bins=40, density=True, alpha=0.6,
                 color=["#2E86AB", "#F39C12"][j], label=name)
axes[1].axvline(theta_true[0], color="#2E86AB", ls="--", lw=2)
axes[1].axvline(theta_true[1], color="#F39C12", ls="--", lw=2)
axes[1].set_title("Marginales posteriores")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Parte 3: La Maldición de la Dimensionalidad en Rejection Sampling

### ¿Por qué rejection sampling no escala?

La tasa de aceptación es $M^{-1}$, y para que $M \cdot q \geq p^{∗}$ en todo el espacio, $M$ debe crecer con la dimensión. En $d$ dimensiones:

- El volumen típico del posterior es proporcional al determinante de la covarianza
- El volumen de la propuesta $q$ cubre una región mucho mayor
- La proporción (posterior / propuesta) cae **exponencialmente** con $d$

Veamos esto empíricamente: ¿cómo cae la tasa de aceptación cuando $\theta$ tiene más dimensiones?

In [ ]:
# Simulate rejection sampling for a simple Gaussian target in d dimensions
# Target: N(0, I_d), Proposal: N(0, (1.5)^2 * I_d)
# True acceptance rate: (1/1.5)^d (geometric decay)

dims = [1, 2, 3, 5, 8, 10, 15, 20]  # <-- CHANGE THIS
n_trials = 100_000

acceptance_rates = []
theoretical_rates = []

for d in dims:
    rng_d = np.random.default_rng(42)
    # Sample from proposal q = N(0, (sigma_q)^2 I)
    sigma_q = 1.5
    theta = rng_d.normal(0, sigma_q, (n_trials, d))

    # log p*(theta) = log N(0,I): -0.5 sum(theta^2)
    log_p = -0.5 * (theta**2).sum(axis=1)
    # log q(theta) = log N(0, sigma_q^2 I)
    log_q = -0.5 * (theta**2).sum(axis=1) / sigma_q**2 - d * np.log(sigma_q)
    # Optimal M: max of p*/q over prior support, roughly (sigma_q)^d
    log_M = d * np.log(sigma_q)  # log of M* = sigma_q^d
    log_alpha = log_p - log_q - log_M

    u = rng_d.uniform(size=n_trials)
    accepted = np.log(u) < log_alpha
    acceptance_rates.append(accepted.mean())
    theoretical_rates.append((1/sigma_q)**d)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.semilogy(dims, acceptance_rates, "o-", color="#2E86AB", lw=2, ms=7, label="Empírica")
ax1.semilogy(dims, theoretical_rates, "--", color="#E94F37", lw=1.5, label=f"Teórica $(1/{sigma_q})^d$")
ax1.set_xlabel("Dimensión $d$")
ax1.set_ylabel("Tasa de aceptación (log)")
ax1.set_title("Rejection Sampling — tasa cae exponencialmente con $d$")
ax1.legend()

# How many proposals needed to get 1000 accepted samples?
n_needed = [1000 / r for r in acceptance_rates]
ax2.semilogy(dims, n_needed, "s-", color="#27AE60", lw=2, ms=7)
ax2.axhline(1_000_000, color="gray", ls=":", lw=1.5, label="1M proposals")
ax2.set_xlabel("Dimensión $d$")
ax2.set_ylabel("Proposals necesarios")
ax2.set_title("Proposals necesarios para 1,000 muestras")
ax2.legend()

plt.tight_layout()
plt.show()

print("\nResumen:")
for d, ar in zip(dims, acceptance_rates):
    print(f"  d={d:3d}: tasa={ar:.4f}, proposals para 1000 muestras: {int(1000/ar):,}")

---
## Parte 4: Importance Sampling para el Posterior

Recordemos el estimador IS: para estimar $\mathbb{E}_{p}[f(\theta)]$:

$$\hat{\mu}^{\text{IS}} = \frac{\sum_i f(\theta_i) \cdot \tilde{w}_i}{\sum_i \tilde{w}_i}, \qquad \theta_i \sim q, \quad \tilde{w}_i = \frac{p^{∗}(\theta_i)}{q(\theta_i)}$$

**Ventaja sobre rejection sampling**: no necesitamos encontrar $M$ ni garantizar $M \cdot q \geq p^{∗}$ — todos los puntos se usan, solo con pesos distintos.

**Limitación**: si $q$ tiene colas ligeras, algunos pesos dominan y la estimación es inestable (como vimos en el notebook 02).

In [ ]:
# IS posterior for logistic regression
n_is = 50_000  # <-- CHANGE THIS
q_mean = np.array([0.0, 1.0])  # proposal mean
q_std = 2.5                     # proposal std  # <-- CHANGE THIS

rng_is = np.random.default_rng(42)
theta_samples = rng_is.normal(q_mean, q_std, (n_is, 2))

# Compute unnormalized log weights
log_post_vals = np.array([log_posterior(th, x_data, y_data, tau) for th in theta_samples])
log_q_vals = (
    norm.logpdf(theta_samples[:, 0], q_mean[0], q_std) +
    norm.logpdf(theta_samples[:, 1], q_mean[1], q_std)
)
log_w = log_post_vals - log_q_vals

# Normalize weights (self-normalized IS)
log_w_stable = log_w - log_w.max()
w = np.exp(log_w_stable)
w_norm = w / w.sum()

# Effective sample size (ESS): measures how many independent samples we effectively have
ess = 1 / (w_norm**2).sum()
print(f"N = {n_is:,} samples, ESS = {ess:.0f} ({ess/n_is*100:.1f}% eficiencia)")

# Weighted estimates of posterior mean
theta0_mean_is = (theta_samples[:, 0] * w_norm).sum()
theta1_mean_is = (theta_samples[:, 1] * w_norm).sum()
print(f"IS: E[θ₀|y] ≈ {theta0_mean_is:.3f}  (θ_true = {theta_true[0]})")
print(f"IS: E[θ₁|y] ≈ {theta1_mean_is:.3f}  (θ_true = {theta_true[1]})")

# Compare IS vs rejection sampling
print(f"\nRS (2000 muestras): θ₀={samples[:,0].mean():.3f}, θ₁={samples[:,1].mean():.3f}")
print(f"IS ({n_is:,} samples, ESS={ess:.0f}): θ₀={theta0_mean_is:.3f}, θ₁={theta1_mean_is:.3f}")

In [ ]:
# Compare IS estimates against ground truth (grid approximation)
# Grid truth for theta0 marginal
theta0_vals = np.linspace(-3, 3, 200)
theta0_marginal_grid = []

for t0 in theta0_vals:
    # Marginalize over theta1 (integrate numerically)
    t1_vals = np.linspace(-1, 5, 100)
    dt1 = t1_vals[1] - t1_vals[0]
    log_p_t1 = np.array([log_posterior(np.array([t0, t1]), x_data, y_data, tau) for t1 in t1_vals])
    theta0_marginal_grid.append(np.exp(log_p_t1 - log_p_t1.max()).sum() * dt1)

theta0_marginal_grid = np.array(theta0_marginal_grid)
theta0_marginal_grid /= theta0_marginal_grid.sum() * (theta0_vals[1] - theta0_vals[0])

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(theta0_vals, theta0_marginal_grid, "k-", lw=2, label="Grid (ground truth)")

# IS weighted histogram
bins = np.linspace(-3, 3, 40)
bin_centers = (bins[:-1] + bins[1:]) / 2
hist_is = np.zeros(len(bins)-1)
for i, th in enumerate(theta_samples[:, 0]):
    b = np.searchsorted(bins, th) - 1
    if 0 <= b < len(hist_is):
        hist_is[b] += w_norm[i]
hist_is /= (bins[1] - bins[0])  # normalize to density

ax.bar(bin_centers, hist_is, width=(bins[1]-bins[0]), alpha=0.5,
       color="#2E86AB", label=f"IS (N={n_is:,}, ESS={ess:.0f})")
ax.axvline(theta_true[0], color="#E94F37", ls="--", lw=2, label=f"θ₀_true = {theta_true[0]}")
ax.set_xlabel("$\\theta_0$")
ax.set_ylabel("Densidad marginal posterior")
ax.set_title("Marginal de $\\theta_0$: IS vs. Grid (ground truth)")
ax.legend()
plt.tight_layout()
plt.show()

---
## Parte 5: El Preview — ¿Por Qué MCMC?

Rejection sampling e IS funcionan en dimensiones bajas. Para modelos con $d > 20$ parámetros (y en machine learning, modelos tienen millones), ambos se vuelven impracticables.

La solución moderna es **Markov Chain Monte Carlo (MCMC)**: en lugar de muestras independientes, construimos una **cadena de Markov** que tiene al posterior como distribución estacionaria. La cadena "explora" el espacio de parámetros de forma inteligente.

```
Rejection Sampling / IS:
  θ₁, θ₂, θ₃, ... i.i.d. desde q → re-ponderar o rechazar
  Problema: la mayoría de la masa de q no está donde importa

MCMC (Metropolis-Hastings):
  θ₁ → θ₂ → θ₃ → ... cadena de Markov
  Cada θₜ₊₁ se propone basándose en θₜ
  Se acepta con probabilidad min(1, p(θ_new)/p(θ_current))
  La cadena se queda donde el posterior es alto
  Garantía: la distribución estacionaria es exactamente p(θ|y)
```

La diferencia clave: MCMC **camina** por el espacio en lugar de muestrear al azar, lo que le permite escalar a dimensiones altas.

In [ ]:
# Simple Metropolis-Hastings for our logistic regression posterior
def metropolis_hastings(log_post_fn, theta_init, step_size, n_steps, seed=None):
    """Simple random-walk Metropolis-Hastings."""
    rng_local = np.random.default_rng(seed)
    d = len(theta_init)
    chain = [theta_init.copy()]
    theta_current = theta_init.copy()
    log_p_current = log_post_fn(theta_current, x_data, y_data, tau)
    n_accepted = 0

    for _ in range(n_steps):
        # Propose a new point (random walk)
        theta_proposed = theta_current + rng_local.normal(0, step_size, d)
        log_p_proposed = log_post_fn(theta_proposed, x_data, y_data, tau)

        # Accept/reject
        log_alpha = log_p_proposed - log_p_current
        if np.log(rng_local.uniform()) < log_alpha:
            theta_current = theta_proposed
            log_p_current = log_p_proposed
            n_accepted += 1
        chain.append(theta_current.copy())

    return np.array(chain), n_accepted / n_steps

theta_init = np.array([0.0, 0.5])
chain, mh_acc_rate = metropolis_hastings(
    log_posterior, theta_init, step_size=0.4, n_steps=10_000, seed=42
)
burnin = 2000
chain_post = chain[burnin:]

print(f"MH aceptación: {mh_acc_rate:.2%} (objetivo: 23-50%)")
print(f"MH E[θ₀|y] ≈ {chain_post[:,0].mean():.3f}  (IS: {theta0_mean_is:.3f}, true: {theta_true[0]})")
print(f"MH E[θ₁|y] ≈ {chain_post[:,1].mean():.3f}  (IS: {theta1_mean_is:.3f}, true: {theta_true[1]})")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Trace plot
axes[0].plot(chain[:500, 0], lw=0.7, color="#2E86AB", label="$\\theta_0$")
axes[0].plot(chain[:500, 1], lw=0.7, color="#E94F37", label="$\\theta_1$")
axes[0].axvline(burnin, color="black", ls=":", lw=1.5)
axes[0].set_title("Trace plot (primeros 500 pasos)")
axes[0].set_xlabel("Paso")
axes[0].legend()

# Chain in 2D
axes[1].contourf(T0, T1, post_grid, levels=15, cmap="Blues", alpha=0.4)
axes[1].plot(chain_post[:300, 0], chain_post[:300, 1], "-", alpha=0.5,
             lw=0.8, color="#E94F37", label="Cadena MH")
axes[1].scatter(chain_post[::50, 0], chain_post[::50, 1], s=15, color="#E94F37", alpha=0.7)
axes[1].plot(*theta_true, "k*", ms=12)
axes[1].set_title("Trayectoria de la cadena sobre el posterior")
axes[1].set_xlabel("$\\theta_0$")
axes[1].set_ylabel("$\\theta_1$")

# Marginal comparison
axes[2].plot(theta0_vals, theta0_marginal_grid, "k-", lw=2, label="Grid (truth)")
axes[2].hist(chain_post[:, 0], bins=40, density=True, alpha=0.6,
             color="#E94F37", label="MCMC")
axes[2].axvline(theta_true[0], color="blue", ls="--", lw=1.5)
axes[2].set_title("Marginal θ₀: MCMC vs. Grid")
axes[2].set_xlabel("$\\theta_0$")
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 🔧 Ejercicios

### Ejercicio 1: Tasa de aceptación vs. dimensión

En la celda de la Parte 3, ya viste cómo la tasa de RS cae con $d$.

1. Cambia `dims` para incluir $d \in \{1, 2, 5, 10, 20, 50\}$
2. Estima cuántos proposals necesitas para obtener 10,000 muestras en cada caso
3. **Compara**: ¿cuántas proposals necesitaría MH en esas mismas dimensiones? (Pista: la tasa de aceptación de MH bien calibrado es ~23-50% en cualquier dimensión)

### Ejercicio 2: Prior informativo

Cambia el prior de $\tau = 3$ a $\tau = 0.5$ (muy informativo, centrado en 0).

1. ¿Cómo cambia la forma del posterior?
2. ¿El posterior converge más o menos rápido al valor verdadero cuando tienes más datos?
3. Con $n_{\text{data}} = 5$ y $\tau = 0.5$: ¿el prior domina o los datos?
4. Con $n_{\text{data}} = 200$ y $\tau = 0.5$: ¿sigue dominando el prior?

In [ ]:
# Space for your experiments
tau_experiment = 0.5  # <-- CHANGE THIS: try 0.5, 1.0, 3.0, 10.0
n_data_experiment = 30  # <-- CHANGE THIS: try 5, 30, 100

# Re-generate data and posterior
x_exp = np.random.default_rng(1).uniform(-2, 2, n_data_experiment)
y_exp = np.random.binomial(1, expit(theta_true[0] + theta_true[1]*x_exp))

log_post_exp_grid = np.array([
    log_posterior(np.array([t0, t1]), x_exp, y_exp, tau_experiment)
    for t0, t1 in zip(T0.ravel(), T1.ravel())
]).reshape(T0.shape)
post_exp_grid = np.exp(log_post_exp_grid - log_post_exp_grid.max())

fig, ax = plt.subplots(figsize=(6, 5))
ax.contourf(T0, T1, post_exp_grid, levels=20, cmap="Blues")
ax.plot(*theta_true, "r*", ms=15, label=f"θ_true")
ax.set_xlabel("$\\theta_0$")
ax.set_ylabel("$\\theta_1$")
ax.set_title(f"Posterior con τ={tau_experiment}, n={n_data_experiment}")
ax.legend()
plt.tight_layout()
plt.show()

---
## Reflexión

| Método | Pros | Contras | Cuándo usar |
|--------|------|---------|-------------|
| **Rejection Sampling** | Simple, muestras exactamente i.i.d. | Tasa cae exponencialmente con $d$ | $d \leq 3$, distribuciones simples |
| **Importance Sampling** | No requiere rechazar, todos los puntos contribuyen | Pesos degenerados si $q$ mal elegida, ESS bajo en alta $d$ | $d \leq 10$, cuando se puede elegir buena $q$ |
| **MCMC (MH, HMC, NUTS)** | Escala a alta $d$, muy general | Correlación entre muestras (burn-in), difícil de diagnosticar convergencia | El estándar para modelos bayesianos reales |

**Conclusión**: MC (en sus formas simples) habilita inferencia bayesiana cuando el posterior es intratable. Para modelos grandes, MCMC es la herramienta correcta — y es la base de librerías modernas como PyMC, Stan y NumPyro.